# ByteTrack + YOLOv5 (modelo swimmer fine-tuned) — Tracking de nadadores en video

Este notebook toma el modelo que entrenamos en `entrenamiento.ipynb` (`best.pt`, guardado en tu Drive) y le agrega **ByteTrack** para hacer *tracking* de nadadores/personas sobre un video — es decir, no solo detectar en cada frame por separado, sino **asignarle un ID consistente a cada nadador** a lo largo del tiempo (sabe que "el nadador 3" en el frame 100 es el mismo que "el nadador 3" en el frame 150).

## Por qué hace falta tracking además de detección
YOLOv5 solo (`detect.py`) detecta objetos **frame por frame, sin memoria** — no sabe si el nadador que ve ahora es el mismo que vio hace 2 segundos. Eso significa que no podés, por ejemplo, contar cuántos nadadores distintos hubo, o seguir la trayectoria de uno en particular, sin que el conteo salte a lo loco. **ByteTrack resuelve eso**: toma las detecciones de YOLOv5, frame a frame, y las conecta entre sí usando posición + movimiento esperado, asignando un ID único a cada objeto que se mantiene mientras lo sigue viendo (incluso tolera que se pierda 1-2 frames por oclusión y lo vuelva a enganchar).

## Qué usamos
- **Detector**: tu `best.pt` (YOLOv5s fine-tuned, clases `persona`/`swimmer`).
- **Tracker**: `ByteTrack` de la librería [`boxmot`](https://github.com/mikel-brostrom/boxmot) — implementación lista para usar, liviana (no necesita reentrenar nada, es puramente algorítmico sobre las cajas que ya detecta YOLOv5).

**Corré esto en Google Colab con GPU**, igual que el entrenamiento.

In [ ]:
#@title 1. Verificar GPU y montar Drive
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 2. Clonar YOLOv5 e instalar dependencias (detección + tracking)
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt
%pip install -q boxmot opencv-python-headless

import torch
print('Torch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())

import boxmot
print('boxmot:', boxmot.__version__ if hasattr(boxmot, '__version__') else 'instalado OK')


In [ ]:
#@title 3. Ubicar el modelo entrenado (best.pt)
import os

# Ajustá esta ruta según cuál corrida quieras usar (la v3 es la más reciente/completa):
WEIGHTS = '/content/drive/MyDrive/yolo_runs/swimmer_finetune_v3/weights/best.pt'

# Si todavía no terminó la v3, podés usar la v1 original:
# WEIGHTS = '/content/drive/MyDrive/yolo_runs/swimmer_finetune/weights/best.pt'
# o el que ya tenías descargado a tu Drive directamente:
# WEIGHTS = '/content/drive/MyDrive/swimmer_best_finetuned_v3.pt'

assert os.path.exists(WEIGHTS), f'No encuentro el modelo en {WEIGHTS} — ajustá la ruta'
print('Modelo encontrado:', WEIGHTS, '-', round(os.path.getsize(WEIGHTS)/1e6, 1), 'MB')


In [ ]:
#@title 3b. Arreglar el checkpoint si fue guardado en Windows (mismo parche que en el entrenamiento)
import pathlib, torch

_original_windows_path = pathlib.WindowsPath
pathlib.WindowsPath = pathlib.PosixPath
try:
    _ = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    print('Checkpoint carga OK, no hace falta arreglar nada.')
finally:
    pathlib.WindowsPath = _original_windows_path


In [ ]:
#@title 4. Cargar el modelo YOLOv5 (listo para inferencia)
import torch

model = torch.hub.load('/content/yolov5', 'custom', path=WEIGHTS, source='local', force_reload=True)
model.conf = 0.35   # umbral de confianza — bajalo (ej. 0.25) si querés priorizar no perderte nadadores (más recall)
model.iou = 0.45    # umbral de IoU para NMS (filtra cajas duplicadas sobre el mismo objeto)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Modelo cargado en:', device)
print('Clases:', model.names)


In [ ]:
#@title 5. Subir el video a analizar
from google.colab import files

print('Elegí el video (.mp4) desde tu PC...')
uploaded = files.upload()
VIDEO_PATH = '/content/' + next(iter(uploaded))
print('Video subido:', VIDEO_PATH)


## Alternativa: usar un video que ya está en tu Drive

Si en vez de subir el video preferís usar uno que ya tenés en Drive (por ejemplo en `VideosSafeSplash/`), saltate la celda 5 y corré esta en su lugar, ajustando la ruta:
```python
VIDEO_PATH = '/content/drive/MyDrive/VideosSafeSplash/tu_video.mp4'
```

In [ ]:
#@title 6. Inicializar el tracker ByteTrack
from boxmot import ByteTrack
import numpy as np

tracker = ByteTrack(
    track_thresh=0.35,   # confianza mínima para considerar una detección "de alta calidad" al trackear
    track_buffer=30,     # cuántos frames tolera que un objeto "desaparezca" (oclusión) antes de darlo por perdido
    match_thresh=0.8,    # qué tan estricta es la asociación entre detecciones consecutivas
)
print('Tracker ByteTrack listo')


In [ ]:
#@title 7. Procesar el video: detectar con YOLOv5 + trackear con ByteTrack
import cv2
import numpy as np
import time

CLASS_NAMES = model.names  # {0: 'persona', 1: 'swimmer'}
COLORS = {0: (0, 200, 0), 1: (0, 140, 255)}  # verde=persona, naranja=swimmer (BGR)

OUTPUT_PATH = '/content/output_tracked.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'No pude abrir el video: {VIDEO_PATH}'

fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))

unique_ids_seen = {0: set(), 1: set()}  # para contar nadadores/personas distintos vistos en todo el video
frame_idx = 0
t0 = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    # --- Detección con YOLOv5 ---
    results = model(frame[:, :, ::-1])  # BGR (OpenCV) -> RGB (lo que espera el modelo)
    det = results.xyxy[0].cpu().numpy()  # columnas: x1, y1, x2, y2, conf, cls

    if det.shape[0] == 0:
        det = np.empty((0, 6))

    # --- Tracking con ByteTrack ---
    # tracker.update espera [x1,y1,x2,y2,conf,cls] y devuelve [x1,y1,x2,y2,track_id,conf,cls,...]
    tracks = tracker.update(det, frame)

    for t in tracks:
        x1, y1, x2, y2, track_id, conf, cls = t[:7]
        cls = int(cls)
        track_id = int(track_id)
        unique_ids_seen.setdefault(cls, set()).add(track_id)

        color = COLORS.get(cls, (255, 255, 255))
        label = f'{CLASS_NAMES[cls]} #{track_id} {conf:.2f}'
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (int(x1), int(y1) - th - 6), (int(x1) + tw + 4, int(y1)), color, -1)
        cv2.putText(frame, label, (int(x1) + 2, int(y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    out.write(frame)

    if frame_idx % 50 == 0:
        elapsed = time.time() - t0
        print(f'Frame {frame_idx}/{total_frames} | {elapsed:.1f}s transcurridos')

cap.release()
out.release()

print()
print(f'Listo -> {OUTPUT_PATH}')
print(f'Personas distintas trackeadas: {len(unique_ids_seen.get(0, set()))}')
print(f'Nadadores distintos trackeados: {len(unique_ids_seen.get(1, set()))}')


In [ ]:
#@title 8. Convertir a H.264 (para que se vea bien embebido en el notebook / navegadores)
!ffmpeg -y -i /content/output_tracked.mp4 -vcodec libx264 -crf 23 /content/output_tracked_h264.mp4 -loglevel error
print('Convertido: /content/output_tracked_h264.mp4')


In [ ]:
#@title 9. Ver el video resultante acá mismo en el notebook
from IPython.display import HTML
from base64 import b64encode

video_path = '/content/output_tracked_h264.mp4'
mp4 = open(video_path, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()

HTML(f'''
<video width=640 controls>
  <source src="{data_url}" type="video/mp4">
</video>
''')


In [ ]:
#@title 10. Guardar y descargar el video final
import shutil

drive_out = '/content/drive/MyDrive/swimmer_tracked_output.mp4'
shutil.copy('/content/output_tracked_h264.mp4', drive_out)
print('Guardado en Drive:', drive_out)

from google.colab import files
files.download('/content/output_tracked_h264.mp4')
